# Introducao ao GARCH(1,1)

Neste notebook, vamos aprender a estimar e interpretar um modelo GARCH(1,1) usando a biblioteca **archbox**.

O modelo GARCH (Generalized Autoregressive Conditional Heteroskedasticity) foi proposto por **Bollerslev (1986)** como
uma extensao do modelo ARCH de Engle (1982). Ele captura a **volatilidade condicional** de series financeiras,
modelando a variancia como funcao de choques passados e variancias passadas.

**Conteudo:**
1. Carregando os dados
2. Analise exploratoria
3. Testando efeitos ARCH
4. Estimando GARCH(1,1)
5. Interpretando os resultados
6. Volatilidade condicional
7. Previsao de volatilidade
8. Exercicios

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Carregando os dados

Vamos trabalhar com retornos diarios do **S&P 500** (dados sinteticos calibrados com parametros realistas).

O dataset contem 2500 observacoes de log-retornos diarios gerados a partir de um processo GARCH(1,1)
com parametros calibrados para o mercado americano:
- $\omega = 1.5 \times 10^{-6}$
- $\alpha = 0.08$
- $\beta = 0.91$
- $\mu = 0.0004$

In [ ]:
# Carregar dados do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

print(f"Periodo: {returns.index[0].date()} a {returns.index[-1].date()}")
print(f"Observacoes: {len(returns)}\n")
print(data.head(10))
print("\n", data.describe())

## 2. Analise exploratoria

Retornos financeiros apresentam **fatos estilizados** bem conhecidos:

1. **Caudas pesadas**: distribuicao leptocurtica (curtose > 3)
2. **Clusters de volatilidade**: periodos de alta volatilidade seguidos por mais alta volatilidade
3. **Ausencia de autocorrelacao nos retornos**: retornos sao aproximadamente i.i.d.
4. **Autocorrelacao nos retornos ao quadrado**: $r_t^2$ apresenta dependencia temporal

Vamos verificar esses fatos nos dados.

In [ ]:
# TODO: Plote a serie de retornos e calcule estatisticas descritivas
# Dicas:
# - Use plot_returns(returns, title='Retornos S&P 500') para plotar a serie
# - Calcule: media, desvio padrao, assimetria (skewness), curtose (kurtosis)
# - Use returns.skew() e returns.kurtosis()
# - Plote o histograma dos retornos com plt.hist()

## 3. Testando efeitos ARCH

Antes de estimar um modelo GARCH, devemos verificar se existe **heteroscedasticidade condicional** nos dados.
O teste **ARCH-LM** de Engle (1982) testa a hipotese nula de que nao ha efeitos ARCH.

**Hipoteses:**
- $H_0$: Nao ha efeitos ARCH (homocedasticidade)
- $H_1$: Ha efeitos ARCH (heteroscedasticidade condicional)

O teste consiste em regredir os residuos ao quadrado em seus valores defasados:

$$\hat{\epsilon}_t^2 = \alpha_0 + \alpha_1 \hat{\epsilon}_{t-1}^2 + \cdots + \alpha_q \hat{\epsilon}_{t-q}^2 + v_t$$

A estatistica do teste e $LM = T \times R^2 \sim \chi^2(q)$.

In [ ]:
# TODO: Use archbox para realizar o teste ARCH-LM
# Dicas:
# - Use arch_lm_test(returns.values, lags=5)
# - O resultado tem atributos: statistic, pvalue
# - Se p-valor < 0.05, rejeita H0 (ha efeitos ARCH)
# - Teste com diferentes lags: 1, 5, 10

## 4. Estimando GARCH(1,1)

O modelo **GARCH(1,1)** especifica a variancia condicional como:

$$\sigma_t^2 = \omega + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2$$

onde:
- $\omega > 0$: intercepto (nivel base de variancia)
- $\alpha_1 \geq 0$: coeficiente ARCH (impacto de choques recentes)
- $\beta_1 \geq 0$: coeficiente GARCH (persistencia da variancia passada)
- $\alpha_1 + \beta_1 < 1$: condicao de estacionariedade

A estimacao e feita por **Maxima Verossimilhanca (MLE)** assumindo distribuicao normal condicional.

In [ ]:
# TODO: Estime um GARCH(1,1) com archbox
# Dicas:
# - Crie o modelo: model = GARCH(returns.values, p=1, q=1)
# - Ajuste: results = model.fit()
# - Exiba o resumo: print(results.summary())

## 5. Interpretando os resultados

Apos estimar o modelo, podemos calcular metricas importantes:

### Persistencia
A **persistencia** e definida como $\alpha + \beta$. Valores proximos de 1 indicam que choques
na volatilidade demoram muito para se dissipar.

### Meia-vida
A **meia-vida** (half-life) e o numero de periodos para um choque de volatilidade decair pela metade:

$$t_{1/2} = \frac{\ln(0.5)}{\ln(\alpha + \beta)}$$

### Variancia incondicional
A **variancia incondicional** (longo prazo) e:

$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}$$

In [ ]:
# TODO: Calcule persistencia (alpha+beta), meia-vida, variancia incondicional
# Dicas:
# - Persistencia: results.persistence()
# - Meia-vida: results.half_life()
# - Variancia incondicional: results.unconditional_variance()
# - Volatilidade incondicional: np.sqrt(results.unconditional_variance())
# - Compare a volatilidade incondicional com o desvio padrao amostral

## 6. Volatilidade condicional

A grande vantagem do GARCH e fornecer uma estimativa da volatilidade **variante no tempo**,
$\sigma_t$, para cada observacao da amostra.

Isso e fundamental para:
- **Gestao de risco**: calculo de VaR (Value-at-Risk)
- **Precificacao de opcoes**: volatilidade como input para Black-Scholes
- **Alocacao de portfolio**: pesos ajustados por volatilidade

In [ ]:
# TODO: Plote a volatilidade condicional estimada
# Dicas:
# - A volatilidade condicional esta em: results.conditional_volatility
# - Use plot_volatility() ou results.plot(which='volatility')
# - Sobreponha os retornos absolutos para comparacao
# - Observe os clusters de volatilidade

## 7. Previsao de volatilidade

O modelo GARCH permite fazer previsoes de volatilidade **h passos a frente**.
Para o GARCH(1,1), a previsao analitica e:

$$E[\sigma^2_{T+h}] = \bar{\sigma}^2 + (\alpha + \beta)^{h-1} (\sigma^2_{T+1} - \bar{\sigma}^2)$$

Note que conforme $h \to \infty$, a previsao converge para a variancia incondicional $\bar{\sigma}^2$.
A velocidade de convergencia depende da persistencia $\alpha + \beta$.

In [ ]:
# TODO: Faca previsao de volatilidade para 10 dias
# Dicas:
# - Use: forecast = results.forecast(horizon=10)
# - forecast['variance'] contem a variancia prevista
# - forecast['volatility'] contem a volatilidade prevista (sqrt da variancia)
# - Plote a previsao e adicione uma linha horizontal para a vol. incondicional
# - Observe como a previsao converge para o nivel de longo prazo

## 8. Exercicios

1. **GARCH(2,1)**: Estime um modelo GARCH(2,1) e compare com o GARCH(1,1) usando AIC/BIC.
   O modelo mais complexo e justificado?

2. **Diagnosticos**: Aplique o teste de Ljung-Box nos residuos padronizados ao quadrado.
   O modelo captura toda a dependencia na volatilidade?

3. **Outro dataset**: Repita a analise usando os dados do Ibovespa (`ibovespa_returns.csv`).
   Os parametros estimados sao diferentes? Por que?

4. **QQ-Plot**: Faca um QQ-plot dos residuos padronizados. Eles seguem distribuicao normal?

In [ ]:
# TODO: Exercicio 1 - Estime GARCH(2,1) e compare com GARCH(1,1)
# Dicas:
# - model_21 = GARCH(returns.values, p=1, q=2)
# - Compare: results.aic vs results_21.aic
# - Menor AIC/BIC indica melhor modelo

## Conclusao

Neste notebook, aprendemos:

- Como testar a presenca de efeitos ARCH nos dados
- Como estimar um modelo GARCH(1,1) com a biblioteca archbox
- Como interpretar os parametros: persistencia, meia-vida e variancia incondicional
- Como extrair e visualizar a volatilidade condicional
- Como fazer previsoes de volatilidade h passos a frente

No proximo notebook, exploraremos modelos **assimetricos** (EGARCH e GJR-GARCH)
que capturam o **efeito alavancagem** — a tendencia de choques negativos aumentarem
mais a volatilidade do que choques positivos.